# Notebook 1: Feature Group Backfill, Pydantic Metadata & Hopsworks Secret Management
**Author:** Guillén Concepción (Senior Data Scientist & MLOps Engineer)

This notebook demonstrates:
1. Reading sensor location parameters (`COUNTRY`, `CITY`, `STREET`, `URL`, `HOPSWORKS_API_KEY`, `AQICN_API_KEY`) using a **Pydantic BaseSettings** object from `.env`.
2. Adding `country`, `city`, `street`, and `url` helper columns to `df_aq`.
3. Joining air quality measurements with weather features using `city` and `timestamp`.
4. Evaluating dataset completeness using `df.isna().sum()`.
5. Registering secrets in **Hopsworks Secrets Store** so downstream notebooks run without local `.env` dependencies.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import hopsworks
from src.config import settings, FEATURE_GROUP_NAME, FEATURE_GROUP_VERSION, HOPSWORKS_PROJECT
from src.data_fetcher import AirQualityDataFetcher, calculate_us_aqi_pm25, check_dataset_completeness
from src.features import generate_feature_pipeline

## 1. Pydantic Settings Configuration Audit

In [ ]:
print("=== Pydantic Sensor Metadata Settings ===")
print(f"• Country : {settings.country}")
print(f"• City    : {settings.city}")
print(f"• Street  : {settings.street}")
print(f"• URL     : {settings.url}")
print(f"• Lat/Lon : ({settings.latitude}, {settings.longitude})")

## 2. Ingest Historical Data & Join Air Quality with Weather by City

In [ ]:
fetcher = AirQualityDataFetcher(city_name=settings.city)

# 1. Fetch Air Quality DataFrame with helper columns
df_aq = fetcher.fetch_historical_air_quality(start_date="2026-02-19", end_date="2026-08-18")
print(f"Air Quality DataFrame columns: {list(df_aq.columns)}")

# 2. Fetch Weather DataFrame using city lat/lon
df_w = fetcher.fetch_historical_weather(start_date="2026-02-19", end_date="2026-08-18")

# 3. Join Air Quality and Weather data on timestamp and city
df_merged = pd.merge(df_aq, df_w, on=["timestamp", "city"], how="outer")
df_merged.sort_values("timestamp", inplace=True)
df_merged.head()

## 3. Dataset Completeness & Missing Data Audit (`isna().sum()`)

In [ ]:
print("=== Missing Data Summary (isna().sum()) ===")
missing_summary = df_merged.isna().sum()
print(missing_summary[missing_summary > 0] if (missing_summary > 0).any() else "✅ Zero missing values detected across dataset!")

## 4. Feature Engineering & Hopsworks Secret Store Backfill

In [ ]:
featured_df = generate_feature_pipeline(df_merged)
featured_df["us_aqi_calculated"] = featured_df["pm2_5"].apply(calculate_us_aqi_pm25)

api_key = settings.hopsworks_api_key
if api_key:
    project = hopsworks.login(api_key_value=api_key, project=HOPSWORKS_PROJECT)
    
    # Store secrets in Hopsworks
    secret_api = project.get_secret_api()
    for sec_k, sec_v in [("COUNTRY", settings.country), ("CITY", settings.city), ("STREET", settings.street), ("URL", settings.url), ("AQICN_API_KEY", settings.aqicn_api_key)]:
        if sec_v:
            try:
                secret_api.create_secret(sec_k, sec_v)
                print(f"🔑 Registered secret '{sec_k}' in Hopsworks.")
            except Exception:
                pass
                
    fs = project.get_feature_store()
    fg = fs.get_or_create_feature_group(
        name=FEATURE_GROUP_NAME,
        version=FEATURE_GROUP_VERSION,
        primary_key=["timestamp", "city"],
        event_time="timestamp",
        description="Hourly IoT Air Quality, Meteorological Features & Sensor Metadata",
        online_enabled=True
    )
    fg.insert(featured_df, write_options={"wait_for_job": False})
    print(f"✅ Successfully backfilled Hopsworks Feature Group '{FEATURE_GROUP_NAME}' v{FEATURE_GROUP_VERSION}!")
else:
    local_path = Path.cwd().parent / "data" / "air_quality_features.parquet"
    local_path.parent.mkdir(exist_ok=True)
    featured_df.to_parquet(local_path, index=False)
    print(f"ℹ️ Saved backfill features locally at {local_path}.")